# IMPORTS

In [ ]:
from __future__ import annotations
import matplotlib.pyplot as plt
import os
from pathlib import Path
from tracking_library import (
    DETECTOR,
    MotionDetector,
    KALMANObject,
    DISTANCE,
    EdgesCollection,
    Edge,
    OriginEdge,
    Matcher,
    SIMILARITY,
    Blob_collection,
    Blob,
    VoidBlob,
    CCSplitter,
    SPLITTER,
    StaticPlot,
    PlotsAggregator,
    VideoCreator,
    TreeViewer,
    FramesCollection,
    Graph,
    list_files,
    load_image,
    stringify_params
)
from IPython.display import Video


# RUNTIME

In [ ]:
cars_files = list_files("sources/cars/frames")
original_bg = Path("sources/cars/bg.jpg")

directory_path = Path("cars_results")

default_video = directory_path / "original" / "video.mp4"
motion_video = directory_path / "detection" / "video.mp4"
matcher_video = directory_path / "matching" / "recap_matcher.mp4"
graph_video = directory_path / "graph" / "video.mp4"

g1 = directory_path / "graph" / "full1.jpg"
g2 = directory_path / "graph" / "full2.jpg"

r1 = directory_path / "paths" / "img1.jpg"
r2 = directory_path / "paths" / "img2.jpg"
r3 = directory_path / "paths" / "img3.jpg"

trails_video_1 = directory_path / "paths" / "video1.mp4"
trails_video_2 = directory_path / "paths" / "video2.mp4"
trails_video_3 = directory_path / "paths" / "video3.mp4"

trails_video_comp_centr = directory_path / "paths" / "video_comp_centr.mp4"
trails_video_comp_kal = directory_path / "paths" / "video_comp_kal.mp4"

In [ ]:
%%script false --no-raise-error

if directory_path.is_dir():
    new = directory_path.with_name(directory_path.name + "_old")
    directory_path.rename(new)
    print(f"{directory_path} already existed, renamed to {new}")
else:
    directory_path.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {directory_path}")

In [ ]:
#empirically,better matrices for the video

KALMANObject.set_kalman_matrices(
    R = [
        [10, 0  ],
        [0,   10]
    ],
    Q = [
        [0.1, 0, 0, 0],
        [0, 0.1, 0, 0],
        [0, 0, 0.1, 0],
        [0, 0, 0, 0.1],
    ]
)

In [ ]:
det = {
        'type': DETECTOR.naive_3D, 
        'tresh': 0.3,
        'fill_holes':5,
        'remove_small':50,
        'bg_rate': 0.05
        }

spl = {
       'type': SPLITTER.naive,
       'remove_inner':True
       }

mat = {
        'type': SIMILARITY.xy,
        's_xy':30,
        'treshold':0.2,
        'max_parents':None
    }

coll = FramesCollection(
    cars_files,original_bg,
    detector_params=det,
    splitter_params=spl,
    matcher_params=mat,
    name=str(directory_path)
).compute()

## ORIGINAL VIDEO

In [ ]:
if not default_video.is_file():
    frames = VideoCreator.create_frames_from_plots(
        coll, 
        [StaticPlot.original_frame],
        video_name=default_video
    )
    default_video.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(default_video), fps=15)
    
Video(str(default_video), embed=True, height=600)


## MOTION DETECTION

In [ ]:
if not motion_video.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll,
        PlotsAggregator.recap_motion_detection,
        params = coll.detector_params
    )
    motion_video.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(motion_video), fps=15)
    
Video(str(motion_video), embed=True, height=600)


## MATCHING

In [ ]:
if not matcher_video.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll,
        PlotsAggregator.recap_matcher,
        minimize=True,
        params = {
            **coll.detector_params,
            **coll.matcher_params,
            **coll.splitter_params
        }
    )
    matcher_video.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(matcher_video), fps=15)
    
Video(str(matcher_video), embed=True, height=600)


## REAL TIME GRAPH

In [ ]:
if not graph_video.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll,
        PlotsAggregator.real_time_graph,
        graph=coll.graph,
        memory=20,
        params = {
            **coll.detector_params,
            **coll.matcher_params,
            **coll.splitter_params
        }
    )
    
    graph_video.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(graph_video), fps=15)
    
Video(str(graph_video), embed=True, height=600)


## graph with N parents

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(25, 5))
StaticPlot.draw_graph(
    coll.graph.get_sub_graph(20,35),
    rects=list(range(0, coll.graph.last_ts, 5)),
    weights=False,ax=ax)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 5))
StaticPlot.draw_graph_mini(coll.graph.get_last_sub_graph(100), ax=ax, title=False, legend=True)
plt.tight_layout()

if not g1.is_file(): plt.savefig(g1, dpi=300)

plt.show()

## graph with 1 parent

In [ ]:
mat['max_parents'] = 1
coll2 = FramesCollection(cars_files,original_bg,
    name=str(directory_path),
    detector_params=det,
    splitter_params=spl,
    matcher_params=mat,
).compute()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(25, 5))
StaticPlot.draw_graph(
    coll2.graph.get_sub_graph(20,35),
    rects=list(range(0, coll2.graph.last_ts, 5)),
    weights=False,ax=ax)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 5))
StaticPlot.draw_graph_mini(coll.graph.get_last_sub_graph(100), ax=ax, title=False, legend=False)
plt.tight_layout()

if not g2.is_file(): plt.savefig(g2, dpi=300)

plt.show()

## TRAILS

In [ ]:
#to improve the visualization
for m in coll.history:  m.curr_CC.detector.image *= 0.5
for m in coll2.history: m.curr_CC.detector.image *= 0.5

### blob, max_parent = 1

In [ ]:
if not trails_video_1.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll2,
        PlotsAggregator.real_time_trails,
        graph=coll2.graph,
        n_chains=1,
        title="Trajectories of the blobs (max 1 parent)"
    )
    
    trails_video_1.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(trails_video_1), fps=15)
    
Video(str(trails_video_1), embed=True, height=600)

In [ ]:
fig1, ax1 = plt.subplots(1, 2, figsize=(10, 4))
for a in ax1:
    StaticPlot.bounding_boxes(coll2.history[-1].curr_CC, legend=False, ids=False, title=False, ax=a)
    a.axis('off')

g2 = coll2.graph
bc2 = g2.get_ending_blobs()
StaticPlot.draw_trail_2d(g2, bc2, trail_kalman=False, max_chains=1, ax=ax1[0], title="Centroid of the Blob", legend=False)
StaticPlot.draw_trail_2d(g2, bc2, trail_centroid=False, max_chains=1, ax=ax1[1], title="Kalman filter of the centroid of the Blob", legend=False)
fig1.suptitle("Trajectories of the blobs (max 1 parent)")
fig1.tight_layout()

if not r1.is_file():  fig1.savefig(r1, bbox_inches='tight', dpi=300)
plt.show()
plt.close(fig1)


### blob, max_parent = N

In [ ]:
if not trails_video_2.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll,
        PlotsAggregator.real_time_trails,
        graph=coll.graph,
        n_chains=1,
        title="Trajectories of the blobs (max N parent)"
    )
    
    trails_video_2.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(trails_video_2), fps=15)
    
Video(str(trails_video_2), embed=True, height=600)

In [ ]:
fig2, ax2 = plt.subplots(1, 2, figsize=(10, 4))
for a in ax2:
    StaticPlot.bounding_boxes(coll.history[-1].curr_CC, legend=False, ids=False, title=False, ax=a)
    a.axis('off')
g1 = coll.graph

bc1 = g1.get_ending_blobs()
StaticPlot.draw_trail_2d(g1, bc1, trail_kalman=False, max_chains=1, ax=ax2[0], title="Centroid of the Blob", legend=False)
StaticPlot.draw_trail_2d(g1, bc1, trail_centroid=False, max_chains=1, ax=ax2[1], title="Kalman filter of the centroid of the Blob", legend=False)
fig2.suptitle("Trajectories of the blobs (max N parent)")
fig2.tight_layout()
 
if not r2.is_file():  fig2.savefig(r2, bbox_inches='tight', dpi=300)
plt.show()
plt.close(fig2)

### Superblobs 

Since in the top part of the image the small blobs collides and merege, the backtrasking procedure is deeply impratical and produces artifacts.
In this case, the problem could be rolved by avoiding the merge of the chains if the blobs are too small (idea not implemented).

In [ ]:
if not trails_video_3.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll,
        PlotsAggregator.real_time_trails,
        graph=coll.graph,
        n_chains=-1,
        title="Trajectories of the Super-Blobs"
    )
    
    trails_video_3.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(trails_video_3), fps=15)
    
Video(str(trails_video_3), embed=True, height=600)

In [ ]:
fig3, ax3 = plt.subplots(1, 2, figsize=(10, 4))
for a in ax3:
    StaticPlot.bounding_boxes(coll.history[-1].curr_CC, legend=False, ids=False, title=False, ax=a)
    a.axis('off')
StaticPlot.draw_trail_2d(g1, bc1, trail_kalman=False, max_chains=-1, ax=ax3[0], title="Centroid of the Super-Blob", legend=False)
StaticPlot.draw_trail_2d(g1, bc1, trail_centroid=False, max_chains=-1, ax=ax3[1], title="Kalman filter of the centroid of the Super-Blob", legend=False)
fig3.suptitle("Trajectories of the Super-Blobs")
fig3.tight_layout()

if not r3.is_file(): fig3.savefig(r3, bbox_inches='tight', dpi=300)
plt.show()
plt.close(fig3)


## FINAL COMPARISONS

In [ ]:
if not trails_video_comp_centr.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll,
        PlotsAggregator.real_time_trail_comparison,
        graph_n=coll.graph,
        graph_1=coll2.graph,
        kalman=False,
    )
    
    trails_video_comp_centr.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(trails_video_comp_centr), fps=15)
    
Video(str(trails_video_comp_centr), embed=True, height=600)

In [ ]:
if not trails_video_comp_kal.is_file():
    frames = VideoCreator.create_frames_from_aggregator(
        coll,
        PlotsAggregator.real_time_trail_comparison,
        graph_n=coll.graph,
        graph_1=coll2.graph,
        kalman=True,
    )
    
    trails_video_comp_kal.parent.mkdir(parents=True, exist_ok=True)
    VideoCreator.save_video(frames, str(trails_video_comp_kal), fps=15)
    
Video(str(trails_video_comp_kal), embed=True, height=600)